In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler,LabelEncoder
from sklearn.model_selection import train_test_split
import pickle

In [4]:
#loading data
data=pd.read_csv(r"D:\Coding Stuff\churnmodeling\Churn_Modelling.csv")
data.sample(5)

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
677,678,15715142,Repina,739,Germany,Male,45,7,102703.62,1,0,1,147802.94,1
5046,5047,15708289,Graham,793,Spain,Male,25,3,100913.57,1,0,0,10579.72,0
6212,6213,15638231,Chung,730,Spain,Female,62,2,0.00,2,1,1,162889.10,0
6524,6525,15743293,Waters,651,Germany,Female,35,1,163700.78,3,1,1,29583.48,1
2535,2536,15578809,Hao,651,Germany,Male,40,1,134760.21,2,0,0,174434.06,1


In [5]:
#preprocessing ,cleaning 
data=data.drop(['RowNumber','CustomerId','Surname'],axis=1)
data.sample(5)

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
5902,694,France,Male,40,9,0.00,2,1,0,40463.03,0
8287,725,France,Female,36,1,118851.05,1,1,1,102747.02,0
8226,804,Spain,Female,38,3,124197.22,1,1,0,74692.06,0
2378,622,France,Female,38,4,98640.74,1,1,1,110457.99,0
775,648,France,Male,33,7,134944.00,1,1,1,117036.38,0


In [6]:
#encoding 
#label encoding
label_encoder_gender=LabelEncoder()
data['Gender']=label_encoder_gender.fit_transform(data['Gender'])

In [7]:
label_encoder_gender

LabelEncoder()

In [8]:
#onehot encoding on geography
from sklearn.preprocessing import OneHotEncoder
onehot_encoder_geo=OneHotEncoder()
geo_encoder=onehot_encoder_geo.fit_transform(data[['Geography']])
geo_encoder

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 10000 stored elements and shape (10000, 3)>

In [9]:
onehot_encoder_geo.get_feature_names_out(['Geography'])

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [10]:
geo_encoder.toarray()

array([[1., 0., 0.],
       [0., 0., 1.],
       [1., 0., 0.],
       ...,
       [1., 0., 0.],
       [0., 1., 0.],
       [1., 0., 0.]])

In [11]:
geo_encoded_df=pd.DataFrame(geo_encoder.toarray(),columns=onehot_encoder_geo.get_feature_names_out(['Geography']))

In [12]:
geo_encoded_df.head(5)

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0
1,0.0,0.0,1.0
2,1.0,0.0,0.0
3,1.0,0.0,0.0
4,0.0,0.0,1.0


In [13]:
data=pd.concat([data.drop('Geography',axis=1),geo_encoded_df],axis=1 )
data.head()

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
0,619,0,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0
1,608,0,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0
2,502,0,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0
3,699,0,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0
4,850,0,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0


In [14]:
data.sample(5)

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain
362,648,0,50,9,102535.57,1,1,1,189543.19,0,1.0,0.0,0.0
7939,720,1,26,10,51962.91,2,1,0,45507.24,0,1.0,0.0,0.0
5410,623,0,28,4,0.00,2,1,0,41227.67,0,1.0,0.0,0.0
8246,663,1,24,7,0.00,2,1,1,166310.82,0,1.0,0.0,0.0
4460,661,1,35,5,0.00,1,1,0,155394.52,0,1.0,0.0,0.0


In [15]:
#divide the data into dependent and independent features
X=data.drop(['Exited'],axis=1)
y=data['Exited']

In [16]:
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

In [17]:
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [18]:
with open('onehot_encoder_geo.pkl','wb') as file:
    pickle.dump(onehot_encoder_geo,file)

with open('label_encoder_gender.pkl','wb') as file:
    pickle.dump(label_encoder_gender,file)

with open('scaler.pkl','wb') as file:
    pickle.dump(scaler,file)

In [19]:
onehot_encoder_geo

,categories,'auto'
,drop,None
,sparse_output,True
,dtype,<class 'numpy.float64'>
,handle_unknown,'error'
,min_frequency,None
,max_categories,None
,feature_name_combiner,'concat'


## ANN Implementation

In [1]:
import tensorflow
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.callbacks import EarlyStopping,TensorBoard
import datetime

In [21]:
input_shape=(X_train.shape[1],)
input_shape

(12,)

In [ ]:
## Build our ANN model
model=Sequential([
    Dense(64,activation='relu',input_shape=(X_train.shape[1],)),## HL1 connected with input layer ## we are using input_shape=(X_train,shape[1],)
    Dense(32,activation='relu'),## HL2
    Dense(1,activation='sigmoid')##output layer , using sigmoid as our function bcz we are using it in binary classification problem
                                ### as here we have churn prediction
])

In [24]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense_1 (Dense)             (None, 64)                832       
                                                                 
 dense_2 (Dense)             (None, 32)                2080      
                                                                 
 dense_3 (Dense)             (None, 1)                 33        
                                                                 
Total params: 2945 (11.50 KB)
Trainable params: 2945 (11.50 KB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


In [28]:
## compile the model
## Adam optimizer comes with default learning rate and some other default values
## similarly binary_cross_entropy or other loss function comes with default values
## we can customise this earlier before using them

import tensorflow
opt=tensorflow.keras.optimizers.Adam(learning_rate=0.01)
loss=tensorflow.keras.losses.BinaryCrossentropy()

model.compile(optimizer=opt,loss="binary_crossentropy",metrics=['accuracy'])

In [29]:
## let's setup temsorboard
log_dir="log/fit/"+datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorflow_callback=TensorBoard(log_dir=log_dir,histogram_freq=1)

In [30]:
## Early Stopping
early_stopping_callback=EarlyStopping(monitor="val_loss",patience=10,restore_best_weights=True)

In [31]:
## Train the model
history=model.fit(X_train,y_train,validation_data=(X_test,y_test),epochs=100,
                  callbacks=[tensorflow_callback,early_stopping_callback]
                  )

Epoch 1/100


250/250 [==============================] - 2s 3ms/step - loss: 0.3935 - accuracy: 0.8381 - val_loss: 0.3655 - val_accuracy: 0.8525
Epoch 2/100
250/250 [==============================] - 0s 2ms/step - loss: 0.3543 - accuracy: 0.8551 - val_loss: 0.3486 - val_accuracy: 0.8575
Epoch 3/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3500 - accuracy: 0.8587 - val_loss: 0.3529 - val_accuracy: 0.8565
Epoch 4/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3425 - accuracy: 0.8583 - val_loss: 0.3381 - val_accuracy: 0.8640
Epoch 5/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3412 - accuracy: 0.8590 - val_loss: 0.3440 - val_accuracy: 0.8480
Epoch 6/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3345 - accuracy: 0.8655 - val_loss: 0.3525 - val_accuracy: 0.8560
Epoch 7/100
250/250 [==============================] - 0s 1ms/step - loss: 0.3345 - accuracy: 0.8596 - val_loss: 0.3422 - val_accuracy: 0.85

In [32]:
## saving our model
model.save('model.h5')

d:\Coding Stuff\venv\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [37]:
##load tensorboard extension
%load_ext tensorboard

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [38]:
%reload_ext tensorboard

In [47]:
%tensorboard --logdir log/fit

Reusing TensorBoard on port 6010 (pid 14708), started 0:00:02 ago. (Use '!kill 14708' to kill it.)

In [42]:
!kill 18316

kill: 18316: No such process


In [52]:
data.iloc[1]

CreditScore             608.00
Gender                    0.00
Age                      41.00
Tenure                    1.00
Balance               83807.86
NumOfProducts             1.00
HasCrCard                 0.00
IsActiveMember            1.00
EstimatedSalary      112542.58
Exited                    0.00
Geography_France          0.00
Geography_Germany         0.00
Geography_Spain           1.00
Name: 1, dtype: float64

In [53]:
data.iloc[1242].to_dict()

{'CreditScore': 696.0,
 'Gender': 1.0,
 'Age': 30.0,
 'Tenure': 4.0,
 'Balance': 114027.7,
 'NumOfProducts': 1.0,
 'HasCrCard': 1.0,
 'IsActiveMember': 1.0,
 'EstimatedSalary': 193716.56,
 'Exited': 0.0,
 'Geography_France': 0.0,
 'Geography_Germany': 1.0,
 'Geography_Spain': 0.0}